### Toy example: Newsvendor problem

#### Imports

In [1]:
import numpy as np
import math
from numpy.random import choice
from sklearn.utils import shuffle
import itertools
from scipy.stats import dirichlet
from tqdm import tqdm
from joblib import Parallel, delayed
import time

from bayesian_dro.Bayesian_DRO_continuous import (
    main_Bayesian_DRO, data_generation, xi_generation, theta_generation
)

#### Class for sampling from the NPL posterior (Assumes a Exponential distribution model)


In [2]:
# NPL class
class npl():
    """This class contains functions to perform NPL inference (for alpha = 0 in the DP prior) for the Exponential distribution model.
    The user supplies parameters:
        X: Data set
        B: number of bootstrap iterations
        p: number of unknown parameters
        loss_fn : string set to 'wll' or 'mmd' to specify either the negative log-lkh or mmd-based loss function 
    """

    def __init__(self, X, B, p, loss_fn='wll'):
        self.B = B
        self.X = X
        self.p = p
        self.loss_fn = loss_fn
        self.n, self.d = self.X.shape

    def draw_samples(self):
        """Draws B samples in parallel from the nonparametric posterior"""

        weights = dirichlet.rvs(np.ones(self.n), size = self.B, random_state = 13)
        samples = np.zeros((self.B,self.p))
        
        if self.loss_fn == 'wll':
            # FIXME n_jobs > 1
            temp = Parallel(n_jobs=1, backend='multiprocessing', max_nbytes=None,batch_size="auto")(delayed(self.WLL)(self.X,weights[i,:]) for i in tqdm(range(self.B)))

            for i in range(self.B):
                  samples[i,:] = temp[i]
                  self.sample = np.array(samples)
        elif self.loss_fn == 'mmd':
            # FIXME implement NPL-MMD here / 1 is just a dummy value for now! 
            self.sample = 1 #np.loadtxt('mmd_samples.txt')

    def WLL(self, data, weights):
        """Get weighted negative log likelihood minimizer, for Exponential distribution model"""

        theta = np.zeros(self.d)
        for i in range(self.n):
            theta += weights[i]*data[i,:]
        return 1/theta

#### Sample observations from the standard Student-t ($\nu = 1$)

In [38]:
# set random seed
myseed = np.random.seed(0)

# Sample observations
# degrees_of_freedom = 2
num_observations = 500
# data_student_t = np.random.standard_t(degrees_of_freedom, size=num_observations)

# DGP is a truncated normal
data = data_generation(num_observations)
data.shape

(500,)

#### Generate NPL posterior samples

In [50]:
n = num_observations
B = 500 # number of bootstrap iterations
p = 1 # numbers of unknown parameters

npl_toy = npl(data.reshape((n,1)),B, p, loss_fn='wll')  # can change loss_fn to wll or mmd
t0 = time.time()
npl_toy.draw_samples()
t1 = time.time()
total = t1-t0
print(f'Total time: {total} seconds')
sample = npl_toy.sample

100%|██████████| 500/500 [00:00<00:00, 1735.96it/s]

Total time: 0.297374963760376 seconds


## Bayesian DRO with NPL Posterior

Sample $\theta$ from the NPL posterior, then apply Bayesian DRO algorithm.

In [51]:
EPSILON = 1.0   # fix epsilon for now
theta = sample
xi = np.zeros([B, num_observations])
for i in range(B):
    xi[i] = xi_generation(theta[i], num_observations)
solution_BDRO_NPL = main_Bayesian_DRO(xi, EPSILON)

/Users/cdellaporta/Desktop/mis-dro-code/bayesian_dro/Bayesian_DRO_continuous.py:107: RuntimeWarning: overflow encountered in exp
  np.mean(np.exp(cost(x, xi[theta_index, :]) / lam))
/Users/cdellaporta/Library/Python/3.9/lib/python/site-packages/scipy/optimize/_numdiff.py:576: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0


## Bayesian DRO

In [52]:
EPSILON = 1.0   # fix epsilon for now
theta = theta_generation(data, B)
xi = np.zeros([B, num_observations])
for i in range(B):
    xi[i] = xi_generation(theta[i], num_observations)
solution_BDRO = main_Bayesian_DRO(xi, EPSILON)

Cost function

In [45]:
b = 4
h = 3
def cost(x, xi):
    return h * np.maximum(0, x - xi) + b * np.maximum(0, xi - x)

In [58]:
# Some new data to approximate true expected cost
myseed = np.random.seed(20)
data_eval = data_generation(10000)
cost_BDRO_NPL = cost(solution_BDRO_NPL, data_eval)
cost_BDRO = cost(solution_BDRO, data_eval)
print(cost_BDRO_NPL.mean(), cost_BDRO.mean())

50.06669998228903 56.6966999822889
